# Phase 6C — Recurrent Neural Networks & LSTMs

**Theory:** RNN, LSTM, GRU — models designed for **sequential data** (time series, text, speech).

**The problem with vanilla RNNs:** Vanishing gradients — can't learn long-range dependencies.
**LSTM solution:** Gates that control what to remember and what to forget.

**Install:** `pip install torch`

---

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset

    TORCH_AVAILABLE = True
    print(f"PyTorch: {torch.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("Install: pip install torch")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

---
## 1. Sequence Data Concepts

**RNN input shape:** `(batch_size, seq_length, input_size)`

- `batch_size`: number of sequences processed simultaneously
- `seq_length`: number of time steps
- `input_size`: number of features at each time step

In [ ]:
# Generate a sine wave dataset — classic sequence prediction task
np.random.seed(42)

t = np.linspace(0, 8 * np.pi, 800)
signal = np.sin(t) + 0.1 * np.random.randn(len(t))

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:200], signal[:200], color="steelblue")
ax.set_title("Sample: Noisy Sine Wave (first 200 steps shown)")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

In [ ]:
def create_sequences(data, seq_len):
    """Turn a 1D time series into (X, y) pairs for supervised sequence learning."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


SEQ_LEN = 30
X_seq, y_seq = create_sequences(signal, SEQ_LEN)

print(f"X shape: {X_seq.shape}  (samples, seq_len)")
print(f"y shape: {y_seq.shape}  (samples,)")

# Train / test split (for time series, do NOT shuffle!)
split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

---
## 2. LSTM for Time Series Prediction

In [ ]:
if TORCH_AVAILABLE:

    class LSTMPredictor(nn.Module):
        def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,  # input shape: (batch, seq, features)
                dropout=0.2,
            )
            self.fc = nn.Linear(hidden_size, output_size)

        def forward(self, x):
            # x shape: (batch, seq_len, input_size)
            lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden_size)
            last_out = lstm_out[:, -1, :]  # take output at last time step
            return self.fc(last_out)

    # Convert to tensors — LSTM expects (batch, seq_len, features)
    X_train_t = torch.FloatTensor(X_train).unsqueeze(-1)  # add feature dim
    y_train_t = torch.FloatTensor(y_train).unsqueeze(-1)
    X_test_t = torch.FloatTensor(X_test).unsqueeze(-1)
    y_test_t = torch.FloatTensor(y_test).unsqueeze(-1)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

    model = LSTMPredictor(input_size=1, hidden_size=64, num_layers=2)
    opt = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    print(model)
    print(f"\nInput shape to LSTM: {X_train_t.shape}  (batch, seq_len, 1 feature)")

In [ ]:
if TORCH_AVAILABLE:
    # Training
    losses = []
    for epoch in range(30):
        model.train()
        epoch_loss = 0
        for xb, yb in train_dl:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_dl))
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1:2d}: Loss={losses[-1]:.5f}")

    # Predictions
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).squeeze().numpy()

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(losses, color="steelblue")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")

    n_show = min(150, len(preds))
    axes[1].plot(y_test[:n_show], label="Actual", color="steelblue", alpha=0.8)
    axes[1].plot(preds[:n_show], label="Predicted", color="coral", alpha=0.8)
    axes[1].set_title("LSTM Predictions vs Actual")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

---
## 3. RNN vs LSTM vs GRU — Comparison

In [ ]:
if TORCH_AVAILABLE:
    # Compare all three architectures
    architectures = {
        "RNN": nn.RNN(input_size=1, hidden_size=64, num_layers=2, batch_first=True),
        "LSTM": nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True),
        "GRU": nn.GRU(input_size=1, hidden_size=64, num_layers=2, batch_first=True),
    }

    dummy_seq = torch.randn(4, 30, 1)  # batch=4, seq_len=30, features=1

    print("Architecture comparison:")
    print(f"{'Model':10s} {'Parameters':>15s} {'Output shape':>20s}")
    print("-" * 50)

    for name, rnn in architectures.items():
        out, _ = rnn(dummy_seq)
        n_params = sum(p.numel() for p in rnn.parameters())
        print(f"{name:10s} {n_params:>15,} {str(out.shape):>20s}")

---
## Summary

| Model | Gates | Parameters | When to Use |
|-------|-------|------------|-------------|
| RNN | None | Fewest | Very short sequences; educational purposes |
| LSTM | 3 (forget, input, output) | Most | Long sequences, complex dependencies |
| GRU | 2 (reset, update) | Middle | Good balance; often matches LSTM with fewer params |

**Practical notes:**
- Always try GRU before LSTM — similar performance, faster training
- For very long sequences (>1000 steps): consider Transformers instead
- `batch_first=True` is strongly recommended — input/output shape is `(batch, seq, features)`
- For time series, **never shuffle** your data when creating train/test splits